In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/players.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2023.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2025.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2024.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/combine_tracking.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/games.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/player_career_successes.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/combine_results.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/player_play.csv


In [2]:
# See exactly what Kaggle gave us
files = []

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        path = os.path.join(dirname, filename)
        files.append({
            "filename": filename,
            "path": path,
            "size_mb": os.path.getsize(path) / 1024**2
        })

file_inventory = pd.DataFrame(files).sort_values("size_mb", ascending=False)

display(file_inventory)

,filename,path,size_mb
2,game_tracking_2025.csv,/kaggle/input/competitions/nfl-big-data-bowl-2...,975.203603
3,game_tracking_2024.csv,/kaggle/input/competitions/nfl-big-data-bowl-2...,684.693234
1,game_tracking_2023.csv,/kaggle/input/competitions/nfl-big-data-bowl-2...,319.923055
8,player_play.csv,/kaggle/input/competitions/nfl-big-data-bowl-2...,140.074770
4,combine_tracking.csv,/kaggle/input/competitions/nfl-big-data-bowl-2...,79.906630
0,players.csv,/kaggle/input/competitions/nfl-big-data-bowl-2...,0.045110
5,games.csv,/kaggle/input/competitions/nfl-big-data-bowl-2...,0.041174
7,combine_results.csv,/kaggle/input/competitions/nfl-big-data-bowl-2...,0.039120
6,player_career_successes.csv,/kaggle/input/competitions/nfl-big-data-bowl-2...,0.013162


In [3]:
# Get the competition data directory
DATA_PATH = os.path.dirname(
    file_inventory.loc[file_inventory["filename"] == "players.csv", "path"].iloc[0]
)

print(DATA_PATH)

/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027


In [4]:
DATA_PATH = "/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"

players = pd.read_csv(f"{DATA_PATH}/players.csv")
combine_results = pd.read_csv(f"{DATA_PATH}/combine_results.csv")
career = pd.read_csv(f"{DATA_PATH}/player_career_successes.csv")
games = pd.read_csv(f"{DATA_PATH}/games.csv")
combine_tracking = pd.read_csv(f"{DATA_PATH}/combine_tracking.csv")
player_play = pd.read_csv(f"{DATA_PATH}/player_play.csv")

print("Loaded!")

Loaded!


In [5]:
for name, df in {
    "players": players,
    "combine_results": combine_results,
    "career": career,
    "games": games,
    "combine_tracking": combine_tracking,
    "player_play": player_play
}.items():
    
    print("\n" + "=" * 80)
    print(f"{name.upper()} — {df.shape}")
    print("=" * 80)
    print(df.columns.tolist())


PLAYERS — (510, 10)
['nfl_id', 'display_name', 'draft_year', 'nfl_position', 'birth_date', 'college_name', 'college_conference', 'draft_round', 'draft_pick_within_round', 'draft_overall_pick']

COMBINE_RESULTS — (510, 18)
['draft_year', 'nfl_id', 'combine_position', 'combine_height', 'combine_weight', 'hand_size', 'arm_length', 'wing_span', 'ten_yd_split', 'forty', 'vertical', 'broad_jump', 'three_cone', 'short_shuttle', 'bench_reps', 'ngs_athleticism_score', 'ngs_college_production_score', 'ngs_final_score']

CAREER — (510, 9)
['nfl_id', 'career_offensive_snaps', 'career_defensive_snaps', 'career_special_teams_snaps', 'career_games_active', 'career_games_started', 'ap_all_pro_1st_team', 'ap_all_pro_2nd_team', 'pro_bowl_original_ballot']

GAMES — (1002, 7)
['game_id', 'game_key', 'season', 'season_type', 'week', 'home_team_abbr', 'visitor_team_abbr']

COMBINE_TRACKING — (463189, 14)
['draft_year', 'event_id', 'nfl_id', 'entity_type', 'time', 'drill_type', 'drill_name', 'attempt', 'x',